# Deploying NVIDIA Nemotron 3.5 Super VL with vLLM

This notebook will walk you through how to run NVIDIA Nemotron 3.5 Super VL with vLLM.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving.

NVIDIA Nemotron 3.5 Super VL is available as two checkpoints:

- **BF16**: [`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16)
- **NVFP4**: [`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4)

**This notebook runs the NVFP4 checkpoint on 1x NVIDIA H200 at W4A16 with MTP speculative decoding.** Commands for BF16 and for other hardware are at the end, under **Additional configurations**.

**Model size:** 120B total parameters, 12B active. A Mamba-2, MoE, and attention hybrid backbone with a [C-RADIO v4-H](https://huggingface.co/nvidia/C-RADIOv4-H) vision encoder and an MLP projector.

**Context length:** up to 1M tokens

**Inputs:** text, images, and video. Output is text.

Prerequisites for this notebook:
- 1x NVIDIA H200 with recent drivers
- [Docker](https://docs.docker.com/engine/install/) with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html)
- Python 3.10+

## Overview

- **Serve** the Nemotron 3.5 Super VL NVFP4 checkpoint using vLLM
- **Query the model** through an OpenAI-compatible API
- **Send images and video** alongside text in the same request
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget
- **Reference commands** for other hardware and precisions

## Table of Contents

1. **Decoding options for this model** - Base, MTP, and LiLiCorr
2. **Multimodal inputs** - Image and video formats, limits, and frame sampling
3. **Environment setup** - Container image, client dependencies, and GPU check
   - Launch on NVIDIA Brev
   - Pull the vLLM Docker image
   - Install notebook client dependencies
   - Verify GPU
4. **OpenAI-compatible server** - Launch vLLM and confirm it is ready
   - Launch the Docker container
   - Configuration reference
   - Start server
   - Wait for the server to be ready
5. **Generate responses** - Chat completions, multimodal inference, and tool calling
   - Client setup
   - Single completion
   - Sequential completions
   - Streamed generation
   - Reasoning
   - Document understanding
   - Video understanding
   - Tool calling
   - Controlling reasoning budget
6. **Cleanup and shutdown** - Free the GPU and reset the kernel
7. **Additional configurations** - Commands for other hardware and precisions
   - 1x GB200
   - 2x GB200 and 4x GB200
   - 2x H100
   - 4x H100
   - 1x DGX Spark
   - 2x A100 and 1x RTX Pro 6000
   - 1M context, text only

## Decoding options for this model

Nemotron 3.5 Super VL can produce tokens three ways. All three serve the same weights and differ only in how many tokens come out of a single forward pass. The base option decodes one token per pass - the other two add speculative decoding, where a cheap draft proposes several tokens ahead, the model verifies them all in a single pass, and every token up to the first mismatch is kept. A rejected token invalidates itself and everything after it, so the payoff depends on how often drafts are right - a bad guess costs compute without producing output.

| Option | Where drafts come from | Extra weights to download |
|---|---|---|
| **Base** (no speculative decoding) | nothing, one token per pass | none |
| **MTP** | a prediction layer inside the checkpoint | none |
| **LiLiCorr** | a separate draft model | LiLiCorr checkpoint |

vLLM supports all three for this checkpoint. Pick exactly one: a server has a single draft path, so the flags conflict at launch rather than stacking. Every configuration here uses MTP except the 1x DGX Spark one under **Additional configurations**, which is the only place LiLiCorr appears.

**Concurrency decides whether speculation pays off.** With few requests in flight the GPU has spare capacity, and speculation spends it to shorten the critical path, which lowers per-request latency. Under heavy load the GPU is already busy producing real tokens, so verifying drafts that end up rejected takes throughput away from queued work. `--max-num-seqs` caps how many requests run at once, so it is the flag that decides which side of that tradeoff you land on.

**Draft length is the main knob.** `--speculative_config.num_speculative_tokens` sets how far ahead to guess, at a value set per configuration below. Raising it improves the best case per step but lowers the odds that the whole run is accepted, and wastes more compute when it is not - lowering it shrinks the win but makes it more consistent.

**Two caches share the GPU.** Attention layers use a KV cache that grows with sequence length, while the Mamba layers keep a fixed-size recurrent state per sequence. `--mamba-ssm-cache-dtype float16` sets what the Mamba state is stored in, `--max-model-len` sets how far the KV cache can grow, and `--gpu-memory-utilization` bounds how much of the GPU the two may claim in total.

This notebook uses **MTP**.

## Multimodal inputs

Nemotron 3.5 Super VL accepts text, images, and video in the same conversation. Multimodal content is sent as a list of parts in the `content` field of a message, following the OpenAI chat format - see **Document understanding** and **Video understanding** below for runnable examples.

| Modality | Request part | Format and limits |
|---|---|---|
| Text | `{"type": "text", "text": ...}` | Up to 1M tokens |
| Image | `{"type": "image_url", "image_url": {"url": ...}}` | RGB JPEG or PNG. 512x512 for simple images, 1536x1024 for documents |
| Video | `{"type": "video_url", "video_url": {"url": ...}}` | MP4, 1-2 minutes |

URLs can be public `http(s)` links, `data:` URLs carrying base64-encoded bytes, or `file://` paths for local media. A `file://` path additionally requires `--allowed-local-media-path` on the server.

> **Note:** Two flags mark a text-only configuration. `--language-model-only` disables the vision tower, and `--chat-template-content-format string` has to be removed for image and video workloads.

**Images per request.** One image for simple and document cases. A multi-document request can batch up to 32 page images in a single call.

**PDFs are not accepted.** Render each page to PNG first - for example with `pymupdf` - and send the pages as images.

### Video frame sampling

Frame sampling is a server-launch setting rather than a request parameter. Pick the profile matching the footage you expect:

| Source resolution | fps | num_frames | Flag |
|---|---|---|---|
| **1080p** | 1 | up to 128 | `--media-io-kwargs '{"video": {"fps": 1, "num_frames": 128}}'` |
| **720p and below** | 2 | up to 256 | `--media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}'` |

The multimodal configurations in this notebook use the 720p profile and add `--video-pruning-rate 0.5`, which prunes half the video tokens using Efficient Video Sampling (EVS).

## Environment setup

### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this notebook on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3KTgHDzfH34Ev5d4AiiMxToFZKP)

### Pull the vLLM Docker image

The model runs inside a vLLM container. Pull it once before starting the server:

```shell
docker pull vllm/vllm-openai:v0.31.0
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `jinja2` renders the chat template it applies.

In [1]:
# Bootstrap pip only if the kernel environment is missing it
import importlib.util, subprocess, sys

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

%pip install -q openai==2.38.0 transformers==5.9.0 "jinja2>=3.1.0" "ipywidgets>=8.0.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Verify GPU

Confirm your GPUs are visible on the host before starting the Docker container.

> **Expected output:** One row per GPU with its name, total memory, and the host driver version. If `nvidia-smi` is not found, the NVIDIA driver is not installed.

In [2]:
# Confirm the GPUs are visible on the host
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

index, name, memory.total [MiB], driver_version
0, NVIDIA H200, 143771 MiB, 580.173.02


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the vLLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --gpus all --ipc=host --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --entrypoint /bin/bash \
  vllm/vllm-openai:v0.31.0
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

Run the `vllm serve` command below from inside this container. The commands in **Additional configurations** at the end run here too.

### Configuration reference

Settings for the configuration this notebook runs.

| Setting | Value | Description |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4` | The NVFP4 checkpoint |
| **Served model name** | `nemotron-3.5-super` | What the client cells send |
| **Hardware** | 1x H200 | |
| **Docker image** | `vllm/vllm-openai:v0.31.0` | |
| **Remote code** | Trusted | The checkpoint ships its own modeling code |
| **Tensor parallel size** | 1 | |
| **Precision** | W4A16, auto-detected from the checkpoint | No flag needed |
| **Max model length** | 114688 | Context window |
| **Max concurrent sequences** | 128 | |
| **Max batched tokens** | 16384 | Caps prefill work per step |
| **GPU memory utilization** | 0.89 | |
| **Block size** | 64, with a 64-token prefix match unit | |
| **CUDA graph capture size** | 256 | |
| **Mamba state cache** | float16, stochastic rounding, 5 philox rounds, flashinfer, align mode | Settings for the recurrent state the Mamba layers keep |
| **Speculative decoding** | MTP, 3 draft tokens | |
| **Video decoding** | 256 frames at 2 fps, pruning rate 0.5 | The 720p-and-below profile with EVS |
| **Reasoning parser** | `nemotron_v3` | Splits thinking from the final answer |
| **Tool parser** | `qwen3_coder`, with auto tool choice | Turns the template's tool XML into OpenAI `tool_calls` |
| **Host / port** | `127.0.0.1:8000` | Matches the client cells |

#### Which of these you actually have to set

The launch command is long because it carries a tuned recipe. The flags fall into three groups.

**Required - the server will not come up or will not behave as this notebook expects without them.**

| Flag | Why |
|---|---|
| `--trust-remote-code` | The checkpoint ships its own modeling code |
| `--tensor-parallel-size` | Must match the number of GPUs the configuration targets, 1 here |
| `--max-model-len` | Without it the model's native 1M context sizes the KV cache and vLLM exits with "KV cache is needed, which is larger than the available KV cache memory" |
| `--reasoning-parser nemotron_v3` | Without it the thinking stays inline in `content` instead of arriving as `reasoning_content` |
| `--tool-call-parser qwen3_coder` with `--enable-auto-tool-choice` | Without them the template's tool XML arrives as plain text instead of `tool_calls` |
| `--served-model-name` | Only so the client cells can say `nemotron-3.5-super`; drop it and pass the full repo ID instead |

**Choices - change these to get different behavior, not more speed.** `--speculative-config.*` picks the decoding path, and the video flags set frame sampling.

**Tuning - part of the validated recipe for this hardware.** Everything else: the `--mamba-*` cache settings, `--block-size`, `--prefix-match-unit`, `--compilation_config.max_cudagraph_capture_size`, `--max-num-seqs`, `--max-num-batched-tokens`, and `--gpu-memory-utilization`. These were measured together, so change them as a set rather than one at a time. A minimal command with only the required flags will serve the model correctly but slower.

### Start server

Run the command below from inside the Docker container terminal. This is the configuration the rest of the notebook is written against.

> **Note:** The first launch takes a while. The progress readout can sit at `0%` for several minutes at a time: first while the weights download from Hugging Face and load, then while the server compiles CUDA kernels. That is expected rather than a hang, so give it time instead of restarting. Later launches reuse the cached weights and compiled kernels and start much faster.

> **Note:** Parser names are backend-specific: vLLM's `nemotron_v3` is `nemotron-v3` in TensorRT-LLM and `nemotron_3` in SGLang.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 1 \
  --max-num-seqs 128 \
  --trust-remote-code \
  --max-model-len 114688 \
  --gpu-memory-utilization 0.89 \
  --max-num-batched-tokens 16384 \
  --block-size 64 \
  --prefix-match-unit 64 \
  --compilation_config.max_cudagraph_capture_size 256 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --enable-mamba-shared-prefix-checkpoint \
  --mamba-cache-mode align \
  --speculative-config.method mtp \
  --speculative-config.num_speculative_tokens 3 \
  --video-pruning-rate 0.5 \
  --media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}' \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3 \
  --host 127.0.0.1 \
  --port 8000
```

### Wait for the server to be ready

In a new terminal, poll `/v1/models`:

```shell
TIMEOUT=${TIMEOUT:-3600}   # seconds to wait before giving up
deadline=$(( $(date +%s) + TIMEOUT ))
ready=0

while [ "$(date +%s)" -lt "$deadline" ]; do
  if curl -sf http://localhost:8000/v1/models | grep -q nemotron-3.5-super; then
    ready=1; break
  fi
  echo "Waiting for server..."; sleep 10
done

if [ "$ready" = 1 ]; then
  echo "Server is ready"
else
  echo "Server did not come up within ${TIMEOUT}s. In the terminal running the server, look for:"
  echo "  access    - 401 or 403 from huggingface.co, or a gated-repo message"
  echo "  download  - the weight download still in progress, or a stalled transfer"
  echo "  memory    - 'KV cache is needed, which is larger than the available KV cache memory'"
  echo "  startup   - a Python traceback, or CUDA out of memory"
fi
```

> **Note:** The loop gives up after `TIMEOUT` seconds, one hour by default. Set `TIMEOUT` before running it to wait longer or fail faster.

Then check what the server is actually serving, and send one short request to confirm it generates:

```shell
curl -s http://localhost:8000/v1/models | python3 -m json.tool

curl -sf http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "nemotron-3.5-super", "prompt": "Hello", "max_tokens": 16}'
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then `Server is ready` once `nemotron-3.5-super` is listed by `/v1/models`. If it times out instead, it prints the four things to check in the server terminal. The final command returns a short JSON completion, confirming the model is loaded and generating.

## Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, video inference, tool calling, and reasoning budget examples.

Recommended sampling for this model is **temperature 1.0** and **top_p 0.95**, which is what the cells use and what the checkpoint's `generation_config.json` sets.

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, raise `max_tokens` so the model has budget left to answer after reasoning.

### Client setup

In [3]:
from openai import OpenAI

# Set this to match the served model name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-super"
BASE_URL = "http://127.0.0.1:8000/v1"
API_KEY = "null"  # vLLM does not require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


def show(response, label=None, show_reasoning=True):
    """Print the answer along with the signals that reveal a bad one."""
    choice = response.choices[0]
    if label:
        print(f"--- {label} ---")

    reasoning = getattr(choice.message, "reasoning", None) or getattr(
        choice.message, "reasoning_content", None
    )
    if show_reasoning and reasoning:
        print("Reasoning:", reasoning)
    if choice.message.content:
        print("Content:", choice.message.content)
    if choice.message.tool_calls:
        print("Tool calls:", choice.message.tool_calls)

    usage = response.usage
    print(
        f"[finish_reason={choice.finish_reason}  "
        f"tokens={usage.prompt_tokens} in / {usage.completion_tokens} out]"
    )
    if choice.finish_reason == "length":
        print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")
    if not (choice.message.content or "").strip() and not choice.message.tool_calls:
        print("WARNING: empty answer - reasoning used the whole budget, raise max_tokens.")


### Single completion

In [4]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Briefly explain: what is vLLM and why is it useful for large model inference?"},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
show(response)

Reasoning: The user wants a brief explanation of vLLM and why it's useful for large model inference. I should cover:

1. What vLLM is (open-source library/framework for LLM serving/inference)
2. Key innovation (PagedAttention - efficient memory management of KV cache)
3. Why it's useful (higher throughput, lower latency, better GPU memory utilization, supports continuous batching, etc.)
4. Keep it concise/brief as requested.

Structure:
- Definition (1-2 sentences)
- Core mechanism (PagedAttention, inspired by virtual memory/OS paging)
- Benefits (throughput, memory efficiency, serving more users/models, continuous batching)
- Use case (production LLM serving)

Avoid being too technical but accurate. No code needed unless helpful, but "briefly explain" suggests prose.

Let's draft:

vLLM is an open-source library and serving engine optimized for high-throughput inference of large language models (LLMs). It addresses a major bottleneck in LLM serving: inefficient GPU memory usage for th

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [5]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=4096,
    )
    show(response, label=f"Q: {prompt}", show_reasoning=False)

--- Q: What is the square root of 144? ---
Content: The square root of 144 is **12**.
[finish_reason=stop  tokens=33 in / 83 out]
--- Q: What is the capital of France? ---
Content: The capital of France is **Paris**.
[finish_reason=stop  tokens=29 in / 34 out]
--- Q: Explain quantum computing in simple terms. ---
Content: Think of a regular computer like a library where every book is either open to page 0 or page 1. It reads those pages one at a time, very quickly.

A quantum computer is like a library where each book can be open to page 0 **and** page 1 at the same time—until you look at it. It can explore many possibilities at once, which lets it solve certain hard problems much faster than ordinary computers.

Here are the key ideas, in plain terms:

**1. Qubits instead of bits**  
Regular computers use **bits**, which are like light switches: off (0) or on (1). Quantum computers use **qubits**. Because of quantum physics, a qubit can be 0, 1, or a blend of both at the same time. Th

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [ ]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
    stream=True,
)

section = None
finish_reason = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

    if chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason

print(f"\n\n[finish_reason={finish_reason}]")
if finish_reason == "length":
    print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")

Reasoning: The user is asking for the first 5 prime numbers.

A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself.

Let's list them:
1. 2 (the only even prime number)
2. 3
3. 5
4. 7
5. 11

(Note: 1 is not a prime number)

So the answer is: 2, 3, 5, 7, 11.

Content: The first 5 prime numbers are: **2, 3, 5, 7, and 11**.

[finish_reason=stop]


### Reasoning

The model supports two modes: **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`.

In [7]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
show(resp)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
show(resp2)

Reasoning on
Reasoning: The user wants a haiku about GPUs.

A haiku is a traditional Japanese poem with:
- 3 lines
- 5 syllables in line 1
- 7 syllables in line 2
- 5 syllables in line 3

Topic: GPUs (Graphics Processing Units)

Possible themes:
- Parallel processing
- Cores, threads, computing
- Gaming, rendering
- Silicon, circuits
- Speed, power
- CUDA, NVIDIA, AMD (maybe too specific?)
- Heat, fans

Let's draft:

Line 1 (5 syllables):
Silicon minds (4) - no
Circuit hearts (3) - no
Parallel cores (4) - no
Thousands of cores (4) - no
Tiny silicon (5) - good
Circuit veins flow (4) - no
Silicon streams (4) - no
Cores in silicon (5) - good

Line 2 (7 syllables):
Processing light (4) - no
Threads of data flow (5) - no
Data streams in parallel (7) - good
Rendering digital dreams (7) - good
Millions of threads compute (7) - good

Line 3 (5 syllables):
Pixels ignite (3) - no
Frames bloom bright (3) - no
Graphics arise (4) - no
Pixels wake now (4) - no
Screens ignite bright (4) - no
Images b

### Document understanding

The model reads text and structure out of document pages, diagrams, and charts.

The cell below sends a diagram from the [Nemotron 3.5 Lightning NVFP4 blog post](https://developer.nvidia.com/blog/developing-nemotron-3-5-lightning-nvfp4-with-qad-using-nvidia-model-optimizer/) and asks for the labeled values back as JSON, each paired with the exact on-image text it was read from.

> **Note:** Each field is checked against the diagram and printed as PASS or FAIL next to the on-image text it was read from. A FAIL prints the expected value, and a value that is right but poorly cited is called out too. Nothing raises, so the cell always runs to the end.


In [9]:
import json
import re

DOC_IMAGE_URL = (
    "https://developer-blogs.nvidia.com/wp-content/uploads/2026/08/"
    "nvfp4-qad-nemotron-3.5-lightning.png"
)

EXPECTED = {
    "title": "QAD for Nemotron 3.5 Lightning NVFP4",
    "step_1_name": "Post-Training Quantization",
    "step_2_name": "Distillation",
    "teacher_precision": "BF16",
    "teacher_size": "66 GB",
    "student_size": "22 GB",
}
FIELDS = list(EXPECTED)


def norm(text):
    """Compare on characters only, so '66 GB' and '66GB' both match."""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


prompt = (
    "Read the diagram and return only JSON, no prose, in this exact shape:\n"
    '{"fields": {' + ", ".join(f'"{f}": "..."' for f in FIELDS) + "}, "
    '"citations": {' + ", ".join(f'"{f}": "..."' for f in FIELDS) + "}}\n"
    "Each citation must be the exact text you read that value from in the image."
)

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": DOC_IMAGE_URL}},
            ],
        }
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

choice = resp.choices[0]
print(
    f"finish_reason: {choice.finish_reason}  |  "
    f"tokens: {resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out"
)

raw = re.sub(r"^```(?:json)?|```$", "", (choice.message.content or "").strip(), flags=re.M)

try:
    result = json.loads(raw.strip())
except json.JSONDecodeError:
    print(raw)
else:
    fields = result.get("fields", {})
    citations = result.get("citations", {})
    for f in FIELDS:
        value = fields.get(f, "")
        citation = citations.get(f) or ""
        want = norm(EXPECTED[f])
        ok = want in norm(value)
        print(f"  {'PASS' if ok else 'FAIL'}  {f:<18} {str(value) or '-'}")
        print(f"        read from {citation!r}")
        if not ok:
            print(f"        expected {EXPECTED[f]!r}")
        elif not citation:
            print("        no citation returned for this value")
        elif want not in norm(citation):
            print("        value matches but its citation does not contain it")


finish_reason: stop  |  tokens: 1507 in / 111 out
  teacher_precision      BF16         <- 'Teacher BF16'
  teacher_size           66 GB        <- '66 GB'
  student_precision      W4A16        <- 'W4A16'
  student_size           22 GB        <- '22 GB'
  student_quantization   NVFP4        <- 'NVFP4'


### Video understanding

The cell below sends a short NVIDIA Omniverse clip to the model via the OpenAI-compatible API and asks it to describe the scene. Use MP4 clips of 1-2 minutes.

Frame sampling comes from `--media-io-kwargs` on the serve command. The configuration above samples 256 frames at 2 fps.

> **Note:** Try video prompts both with and without thinking enabled, since which one works better varies by clip. The cell below disables it for a faster, more direct answer.

In [10]:
video_url = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what happens in this video."},
                {"type": "video_url", "video_url": {"url": video_url}},
            ],
        }
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

show(resp)

Content: The video displays a snowy scene with a house and trees in a 3D modeling software interface.
[finish_reason=stop  tokens=1741 in / 22 out]


### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`, then run the function and hand its result back so the model can answer the user.

The chat template emits tool calls as `<tool_call><function=name><parameter=...>` XML, which is what the `qwen3_coder` tool parser converts into OpenAI `tool_calls`.

> **Note:** When tool calling with reasoning enabled, pass `"force_nonempty_content": true` inside `chat_template_kwargs`.

In [11]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calculate the tip amount for a bill",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                "tip_percentage": {"type": "integer", "description": "The percentage of tip to apply"},
            },
            "required": ["bill_total", "tip_percentage"],
        },
    },
}]

messages = [{"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "force_nonempty_content": True}},
)

choice = completion.choices[0]
show(completion)

Reasoning: The user wants to calculate a tip for a bill.
Bill total: $50
Tip percentage: 15%

I need to use the `calculate_tip` function.
Parameters:
- `bill_total`: 50
- `tip_percentage`: 15

Let's call the function.
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-a83ada6c400be616', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]
[finish_reason=tool_calls  tokens=339 in / 106 out]


In [12]:
import json

def calculate_tip(bill_total, tip_percentage):
    """Return the tip and the final total for a bill."""
    tip = round(bill_total * tip_percentage / 100, 2)
    return {"tip": tip, "total": round(bill_total + tip, 2)}

# Map schema names to real functions, then run whichever one the model picked
tool_functions = {"calculate_tip": calculate_tip}

call = choice.message.tool_calls[0]
result = tool_functions[call.function.name](**json.loads(call.function.arguments))

# Hand the result back so the model can answer the user
followup = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages + [
        choice.message,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ],
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Tool result:", result)
show(followup)

Tool result: {'tip': 7.5, 'total': 57.5}
Content: Based on your $50 bill, a 15% tip would be **$7.50**, making the total amount **$57.50**.
[finish_reason=stop  tokens=478 in / 36 out]


### Controlling reasoning budget

`ThinkingBudgetClient` below caps how long the model reasons. It asks for at most `reasoning_budget` tokens of reasoning, closes the reasoning trace if the model has not closed it, then sends a second request for the answer using the tokens left over. If the model already answered within the reasoning budget, that answer is returned as is and no second request is sent.

> **Note:** `max_tokens` must be larger than `reasoning_budget`, since the answer is generated from the remainder.

In [14]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name_or_path, trust_remote_code=True
        )
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        choice = response.choices[0]
        reasoning_content = choice.message.reasoning or ""

        # The model may finish inside the reasoning budget - return that answer
        # rather than spending a second request regenerating it
        if choice.finish_reason == "stop" and (choice.message.content or "").strip():
            return {
                "reasoning_content": reasoning_content.strip(),
                "content": choice.message.content,
                "finish_reason": choice.finish_reason,
            }

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # Copy before extending so the caller's list is left untouched
        messages = list(messages) + [{"role": "assistant", "content": reasoning_content}]
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().removesuffix("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [ ]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4"  # use actual HF model ID for tokenizer
)

In [16]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])
print(f"\n[finish_reason={resp['finish_reason']}]")
if resp["finish_reason"] == "length":
    print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")

Reasoning: The user wants a haiku about GPUs. A haiku is a 3-line poem with a 5-7-5 syllable structure.

Key concepts about GPUs:
- Graphics processing unit
- Parallel processing
- Cores/threads
- Gaming, rendering, AI, crypto
- Silicon, chips, heat, fans
- Speed, compute, matrix math

Let's draft:

Line 1 (5 syllables):
Thou-sand cores a-wake (5)
Si-li-con minds rise (5)
Cores burn bright and fast (5)

Line 2 (7 syllables):
Pa-ral-lel.
Content: Thousand cores awake,
Parallel compute streams,
Rendering the light.


## Cleanup and shutdown

To free resources after this notebook:

1. In the terminal running `vllm serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Restart the kernel if needed to ensure a clean state.

## Additional configurations

Commands for the hardware and precision combinations this notebook does not run. All of them run from inside the same Docker container.

### 1x GB200

#### NVFP4

**MTP, text only**

Draft length is 5 on Blackwell.

```shell
vllm serve --model nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --max-num-seqs 512 \
  --max-model-len 114688 \
  --enable-prefix-caching \
  --max-num-batched-tokens 32768 \
  --prefix-match-unit 16 \
  --enable-mamba-shared-prefix-checkpoint \
  --async-scheduling \
  --kernel-config '{"linear_backend_per_quant": {"fp8_w8a8": "torch"}}' \
  --compilation-config '{"pass_config": {"fuse_attn_quant": true}, "use_inductor_graph_partition": true, "cudagraph_mode": "FULL_AND_PIECEWISE"}' \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --chat-template-content-format string \
  --mamba-cache-mode align \
  --video-pruning-rate 0.5 \
  --media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}' \
  --enable-auto-tool-choice \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

Remove `--chat-template-content-format string` for image and video workloads.

### 2x GB200 and 4x GB200

#### BF16, 2x GB200

**MTP, multimodal**

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 2 \
  --trust-remote-code \
  --max-model-len 114688 \
  --async-scheduling \
  --allowed-local-media-path=/ \
  --max-num-seqs=32 \
  --limit-mm-per-prompt '{"video": 1, "image": 1}' \
  --video-pruning-rate 0.5 \
  --media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}' \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3 \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5
```

#### NVFP4, 4x GB200, text only

**MTP**

```shell
vllm serve --model nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 4 \
  --enable-expert-parallel \
  --max-num-seqs 512 \
  --trust-remote-code \
  --max-model-len 28672 \
  --no-enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --chat-template-content-format string \
  --mamba-cache-mode align \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

#### BF16, 4x GB200, text only

**MTP**

The same measurement point in BF16. The mamba state cache is float32 here, with no stochastic rounding.

```shell
vllm serve --model nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 4 \
  --enable-expert-parallel \
  --max-num-seqs 512 \
  --trust-remote-code \
  --max-model-len 28672 \
  --no-enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float32 \
  --chat-template-content-format string \
  --mamba-cache-mode align \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

### 2x H100

#### NVFP4 / W4A16

**MTP, multimodal**

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 2 \
  --max-num-seqs 128 \
  --trust-remote-code \
  --max-model-len 114688 \
  --gpu-memory-utilization 0.89 \
  --max-num-batched-tokens 16384 \
  --block-size 64 \
  --prefix-match-unit 64 \
  --compilation_config.max_cudagraph_capture_size 256 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --enable-mamba-shared-prefix-checkpoint \
  --mamba-cache-mode align \
  --speculative-config.method mtp \
  --speculative-config.num_speculative_tokens 3 \
  --video-pruning-rate 0.5 \
  --media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}' \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

### 4x H100

#### BF16


**MTP, multimodal**

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16 \
  --served-model-name nemotron-3.5-super \
  --tensor-parallel-size 4 \
  --trust-remote-code \
  --max-model-len 114688 \
  --gpu-memory-utilization=0.85 \
  --allowed-local-media-path=/ \
  --max-num-seqs=32 \
  --limit-mm-per-prompt '{"video": 1, "image": 1}' \
  --video-pruning-rate 0.5 \
  --media-io-kwargs '{"video": {"fps": 2, "num_frames": 256}}' \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3 \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 3
```

### 1x DGX Spark

Both configurations target minimum latency.

#### NVFP4

**LiLiCorr, 7 draft tokens**

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --max-num-seqs 1 \
  --trust-remote-code \
  --max-model-len 114688 \
  --gpu-memory-utilization 0.75 \
  --max-num-batched-tokens 8192 \
  --speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4-LiLiCorr \
  --speculative_config.num_speculative_tokens 7 \
  --speculative_config.method dflash \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --enable-mamba-shared-prefix-checkpoint \
  --prefix-match-unit 16 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

**MTP, 5 draft tokens, text only**

Remove `--chat-template-content-format string` for image and video workloads.

```shell
vllm serve --model nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --max-num-seqs 2 \
  --trust-remote-code \
  --max-model-len 114688 \
  --gpu_memory_utilization 0.7 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mamba-cache-mode align \
  --prefix-match-unit 16 \
  --enable-mamba-shared-prefix-checkpoint \
  --chat-template-content-format string \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

### 2x A100 and 1x RTX Pro 6000

Set the draft length to 5 for low latency.

#### NVFP4 / W4A16, 2x A100

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --tensor-parallel-size 2 \
  --pipeline-parallel-size 1 \
  --data-parallel-size 1 \
  --max-model-len 8192 \
  --max-num-seqs 512 \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

#### NVFP4 / W4A4, 1x RTX Pro 6000

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --tensor-parallel-size 1 \
  --pipeline-parallel-size 1 \
  --data-parallel-size 1 \
  --max-model-len 16384 \
  --max-num-seqs 128 \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

### 1M context, text only

These serve the full 1048576-token context window. All four pass `--language-model-only`, which disables the vision tower, so they do not accept image or video input.

#### NVFP4, 1x GB200

```shell
export VLLM_ALLOW_LONG_MAX_MODEL_LEN=1

vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --language-model-only \
  --max-model-len 1048576 \
  --max-num-seqs 32 \
  --async-scheduling \
  --enable-prefix-caching \
  --prefix-match-unit 16 \
  --enable-mamba-shared-prefix-checkpoint \
  --mamba-cache-mode align \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

#### BF16, 4x GB200

```shell
export VLLM_ALLOW_LONG_MAX_MODEL_LEN=1

vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --language-model-only \
  --tensor-parallel-size 4 \
  --max-model-len 1048576 \
  --max-num-seqs 32 \
  --async-scheduling \
  --enable-prefix-caching \
  --prefix-match-unit 16 \
  --enable-mamba-shared-prefix-checkpoint \
  --mamba-cache-mode align \
  --mamba-backend flashinfer \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 5 \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

#### NVFP4, 2x H100

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --served-model-name nemotron-3.5-super \
  --trust-remote-code \
  --language-model-only \
  --enable-expert-parallel \
  --block-size 64 \
  --prefix-match-unit 64 \
  --enable-mamba-shared-prefix-checkpoint \
  --mamba-cache-mode align \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --chat-template-content-format string \
  --compilation_config.max_cudagraph_capture_size 16 \
  --max-num-seqs 16 \
  --tensor-parallel-size 2 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3
```

#### BF16, 4x H100

The same command with these three values changed:

```shell
  --compilation_config.max_cudagraph_capture_size 2 \
  --max-num-seqs 2 \
  --tensor-parallel-size 4 \
```